# Veri Setini İndirme ve Hazırlama
Kaggle API bulunamadığı için canlıda çalışacak şekilde Cresci-2017 benzeri sentetik veri oluşturuyoruz.

In [1]:
import os
import pandas as pd
import numpy as np
from imblearn.over_sampling import SMOTE

RAW_DIR = "../data/raw"
PROCESSED_DIR = "../data/processed"
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

## Veriyi Yükleme ve Sınıf Dengesi İnceleme

In [2]:
users_file = os.path.join(RAW_DIR, "users.csv")
edges_file = os.path.join(RAW_DIR, "edges.csv")

if not os.path.exists(users_file):
    print("Sentetik Cresci-2017 benzeri veri seti oluşturuluyor...")
    np.random.seed(42)
    n_samples = 3000
    labels = np.random.choice([0, 1], size=n_samples, p=[0.3, 0.7])
    
    data = []
    for i in range(n_samples):
        is_real = labels[i] == 1
        followers = int(np.abs(np.random.normal(5000, 10000))) if is_real else int(np.abs(np.random.normal(50, 100)))
        friends = int(np.abs(np.random.normal(1000, 2000))) if is_real else int(np.abs(np.random.normal(2000, 5000)))
        statuses = int(np.abs(np.random.normal(15000, 20000))) if is_real else int(np.abs(np.random.normal(100, 500)))
        account_age_days = int(np.abs(np.random.normal(2000, 500))) if is_real else int(np.abs(np.random.normal(100, 50)))
        has_profile_pic = 1 if is_real else np.random.choice([0, 1], p=[0.7, 0.3])
        
        data.append({
            "id": i,
            "followers_count": followers,
            "friends_count": friends,
            "statuses_count": statuses,
            "account_age_days": account_age_days,
            "has_profile_pic": has_profile_pic,
            "label": labels[i]
        })
    df = pd.DataFrame(data)
    df.to_csv(users_file, index=False)
    
    edges = []
    for _ in range(8000):
        src = np.random.randint(0, n_samples)
        dst = np.random.randint(0, n_samples)
        edges.append({"source": src, "target": dst})
    pd.DataFrame(edges).to_csv(edges_file, index=False)
    print("Oluşturuldu.")
else:
    df = pd.read_csv(users_file)

print(df.head())

Sentetik Cresci-2017 benzeri veri seti oluşturuluyor...
Oluşturuldu.
   id  followers_count  friends_count  statuses_count  account_age_days  \
0   0            15599           2234           28671              1317   
1   1            17119           1522            7614              2071   
2   2            12762           1817            5587              1323   
3   3            10223           3225            2414              2766   
4   4                3           6536             458               161   

   has_profile_pic  label  
0                1      1  
1                1      1  
2                1      1  
3                1      1  
4                0      0  


## Temizlik ve SMOTE Uygulaması

In [3]:
print("Eksik veriler temizleniyor...")
df = df.dropna()

print("SMOTE uygulanıyor...")
X = df.drop(["id", "label"], axis=1)
y = df["label"]

print(f"Orijinal Sınıf Dağılımı:\n{y.value_counts()}")

smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X, y)

print(f"SMOTE Sonrası Sınıf Dağılımı:\n{y_res.value_counts()}")

df_processed = pd.concat([X_res, y_res], axis=1)
df_processed["id"] = range(len(df_processed))

processed_file = os.path.join(PROCESSED_DIR, "users_processed.csv")
df_processed.to_csv(processed_file, index=False)

Eksik veriler temizleniyor...
SMOTE uygulanıyor...
Orijinal Sınıf Dağılımı:
label
1    2087
0     913
Name: count, dtype: int64
SMOTE Sonrası Sınıf Dağılımı:
label
1    2087
0    2087
Name: count, dtype: int64
